# ⚡ `JEV-Set-150M` (`149,936,905` Params) — Autonomous Colab Pro Runner

Official Training, Calibration, Model Souping & Benchmark Evaluation Pipeline for the **Jev Decision Index v0.4.1**.

- **Parameter Budget**: Strict $\le 150\text{M}$ ceiling (Serves exactly `149,936,905` params; `63,095` margin).
- **Architecture**: `cross-encoder/ettin-reranker-150m-v1` backbone + `EquivariantSetInteractionHead`.
- **Pipeline**: Stage 1 Broad OOD $\rightarrow$ Stage 2 Multi-Teacher Distillation $\rightarrow$ Stage 3 Gold Linked-Case Alignment $\rightarrow$ 3-Seed Model Souping + L-BFGS calibration $\rightarrow$ Official Evaluation.

### Configuration & Run Mode
- `RUN_MODE = 'full'`: Automatically executes the entire pipeline (pilot check, full training stages, souping, evaluation).
- `RUN_MODE = 'pilot'`: Fast hardware and real-weights verification test (~5-10 minutes).

In [ ]:
# ⚙️ PIPELINE MODE SELECTION
RUN_MODE = 'full'  # Options: 'pilot' or 'full'
print(f'✅ Configured RUN_MODE: {RUN_MODE}')

### 🌐 Optional: Connect Antigravity Terminal Directly to this Colab Runtime
If you want Antigravity to take direct terminal control of this GPU:
1. Run the cell below.
2. Click the authorization link (`https://github.com/login/oauth/device`) and enter the 8-character code.
3. Tell Antigravity the tunnel name in chat — Antigravity will immediately connect its terminal!

*If you just want Colab to run autonomously, skip this cell and simply click **Runtime -> Run all**!*

In [ ]:
# 🔌 Start VS Code Remote Tunnel to connect Antigravity directly
!curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' | tar -xz
!./code tunnel --accept-server-license-terms

## Step 1: Detect GPU & Hardware Acceleration

In [ ]:
import os
import sys
import torch

print('=== Hardware Detection ===')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f'✅ GPU Detected: {device_name}')
    print(f'✅ Dedicated VRAM: {vram_gb:.2f} GB')
    print(f'✅ Native bfloat16: {bf16_supported}')
else:
    print('⚠️ WARNING: No CUDA GPU detected. Please go to Runtime -> Change runtime type -> Select GPU (A100 or L4)!')

## Step 2: Clone Repository & Mount Google Drive

In [ ]:
# Mount Google Drive to preserve checkpoints & final models
from google.colab import drive
try:
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/jev_set_150m_checkpoints', exist_ok=True)
    print('✅ Google Drive mounted successfully.')
except Exception as e:
    print('ℹ️ Google Drive mounting skipped or not authorized:', e)

# Clone repository
%cd /content
!rm -rf jev-set-150m
!git clone https://github.com/kassimyassin/jev-set-150m.git
%cd /content/jev-set-150m
print('✅ Repository cloned and ready at /content/jev-set-150m')

## Step 3: Install Pinned Dependencies & Evaluation Kit

In [ ]:
%cd /content/jev-set-150m
!pip install -q -e .
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn optional, falling back to SDPA'

# Install official decision-index evaluation suite
!rm -rf /content/decision-index
!git clone https://github.com/apolinario/decision-index.git /content/decision-index
!pip install -q -e /content/decision-index
print('✅ Dependencies & official evaluation engine installed successfully.')

## Step 4: Run Real-Weights Verification & Test Suite

In [ ]:
%cd /content/jev-set-150m
print('--- Running Parameter Count & Weight Verification ---')
!python scripts/verify_real_weights.py

print('--- Running Unit Test Suite ---')
!pytest tests/ -q

## Step 5: Execute Training Pipeline

In [ ]:
%cd /content/jev-set-150m
print(f'🚀 Launching Pipeline in mode: {RUN_MODE}')
!bash scripts/run_colab_pro.sh {RUN_MODE}

## Step 6: Backup Artifacts to Google Drive & Summary

In [ ]:
import shutil

dest_dir = '/content/drive/MyDrive/jev_set_150m_checkpoints'
if os.path.exists('/content/drive/MyDrive') and os.path.exists('/content/jev-set-150m/checkpoints'):
    print(f'📦 Backing up checkpoints to {dest_dir}...')
    !cp -ru /content/jev-set-150m/checkpoints/* {dest_dir}/
    print('✅ Backup complete!')

print('🎉 JEV-Set-150M Pipeline Run Complete!')